# Kaismart Solutions S.A.S. — Transformación (Metodología Medallion)

**Parte 7 del ejercicio.** A partir de los problemas de calidad detectados en [`01_extraccion_y_eda.ipynb`](01_extraccion_y_eda.ipynb), se implementa el proceso de limpieza, transformación e integración siguiendo la metodología **Medallion**:

- **Bronze**: datos crudos, tal como llegan de las fuentes, solo con metadatos de trazabilidad de la ingesta.
- **Silver**: limpieza y estandarización **independiente** por fuente → `df_ventas_transformado`, `df_logistica_transformado`.
- **Gold**: integración de ambas fuentes por `pedido_id` + datamarts listos para análisis de negocio.

Cada decisión de imputación/estandarización está documentada en el código (`src/kaismart_etl/transform/silver.py`) y se resume aquí mediante los `reporte` que retorna cada función — no se realiza ninguna imputación automática sin justificar el motivo.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

from kaismart_etl import quality
from kaismart_etl.extract import extract_logistica, extract_ventas
from kaismart_etl.transform import bronze, gold, silver

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

df_ventas = extract_ventas()
df_logistica = extract_logistica()
print(f"df_ventas: {df_ventas.shape}  |  df_logistica: {df_logistica.shape}")

2026-09-30 22:24:20 | INFO     | kaismart_etl.db | Creando engine MySQL -> host=107.180.112.11 db=clientes tabla=ventas
2026-09-30 22:24:22 | INFO     | kaismart_etl.extract | Consultando todos los registros de la tabla 'ventas'...
2026-09-30 22:24:22 | INFO     | kaismart_etl.extract | Extraccion de ventas completada: 5000 registros
2026-09-30 22:24:22 | INFO     | kaismart_etl.extract | Conexion a MySQL cerrada correctamente.
2026-09-30 22:24:22 | INFO     | kaismart_etl.extract | Leyendo archivo Excel: D:\Cursos\Phyton\kaismart-etl\data\raw\kaismart_eventos_logisticos (1).xlsx
2026-09-30 22:24:31 | INFO     | kaismart_etl.extract | Extraccion de eventos logisticos completada: 50000 registros


df_ventas: (5000, 17)  |  df_logistica: (50000, 14)


## Capa Bronze

Se persiste una copia inmutable de los datos crudos (sin ninguna limpieza), agregando solo metadatos de trazabilidad (`_fuente`, `_fecha_ingesta`). Esto permite reprocesar Silver/Gold en el futuro sin necesidad de volver a consultar las fuentes originales.

In [2]:
ruta_bronze_ventas = bronze.guardar_bronze(df_ventas, "ventas", fuente="mysql.clientes.ventas")
ruta_bronze_logistica = bronze.guardar_bronze(
    df_logistica, "logistica", fuente="kaismart_eventos_logisticos.xlsx"
)
print("Bronze ventas:", ruta_bronze_ventas)
print("Bronze logistica:", ruta_bronze_logistica)

2026-09-30 22:24:39 | INFO     | kaismart_etl.transform.bronze | Bronze guardado: D:\Cursos\Phyton\kaismart-etl\data\bronze\ventas.csv (5000 registros)
2026-09-30 22:24:40 | INFO     | kaismart_etl.transform.bronze | Bronze guardado: D:\Cursos\Phyton\kaismart-etl\data\bronze\logistica.csv (50000 registros)


Bronze ventas: D:\Cursos\Phyton\kaismart-etl\data\bronze\ventas.csv
Bronze logistica: D:\Cursos\Phyton\kaismart-etl\data\bronze\logistica.csv


## Capa Silver — `df_ventas_transformado`

Decisiones de limpieza aplicadas (ver `silver.limpiar_ventas`):

1. Se eliminan filas **completamente duplicadas** y duplicados de `id_venta` (llave primaria) — conservando el primer registro.
2. Filas sin `pedido_id` se descartan (sin esa llave no es posible integrar con logística).
3. `fecha_venta` se parsea a `datetime`; fechas inválidas quedan como `NaT` (no se inventa una fecha).
4. `ciudad`, `canal`, `categoria` se **estandarizan** contra un catálogo controlado (mayúsculas/tildes/espacios).
5. `metodo_pago` nulo → `"NO INFORMADO"` (sin significado de negocio claro para el nulo).
6. `cantidad` y `precio_unitario` inválidos (≤ 0) → se imputan con la **mediana por categoría** (variables con outliers, more robusta que la media).
7. `valor_descuento` nulo → se imputa con **0** (representa "sin descuento", no una estimación).
8. `valor_bruto` y `valor_neto` se **recalculan** a partir de `cantidad`, `precio_unitario` y `valor_descuento` para garantizar consistencia aritmética.
9. `calificacion_cliente`: los nulos **se conservan** (el cliente no calificó, es válido); solo se corrigen valores fuera de rango 1-5.

In [3]:
df_ventas_transformado, reporte_ventas = silver.limpiar_ventas(df_ventas)

print("Reporte de limpieza df_ventas -> df_ventas_transformado:")
for k, v in reporte_ventas.items():
    print(f"  {k}: {v}")

print(f"\nShape antes: {df_ventas.shape}  ->  Shape despues: {df_ventas_transformado.shape}")
df_ventas_transformado.head()

Reporte de limpieza df_ventas -> df_ventas_transformado:
  filas_entrada: 5000
  duplicados_completos_eliminados: 0
  duplicados_id_venta_eliminados: 0
  filas_sin_pedido_id_eliminadas: 0
  fechas_venta_invalidas: 0
  estandarizacion_ciudad: {'valores_originales_no_nulos': 5000, 'valores_reconocidos': 5000, 'valores_no_reconocidos': 0, 'detalle_no_reconocidos': {}}
  estandarizacion_canal: {'valores_originales_no_nulos': 5000, 'valores_reconocidos': 5000, 'valores_no_reconocidos': 0, 'detalle_no_reconocidos': {}}
  estandarizacion_categoria: {'valores_originales_no_nulos': 5000, 'valores_reconocidos': 5000, 'valores_no_reconocidos': 0, 'detalle_no_reconocidos': {}}
  cantidad_imputada_mediana: 0
  precio_unitario_imputado_mediana: 0
  valor_descuento_nulos_imputados_0: 0
  valor_descuento_negativos_corregidos: 0
  valor_bruto_inconsistentes_recalculados: 0
  calificacion_fuera_de_rango_a_nulo: 0
  calificacion_nulos_conservados: 2172
  filas_salida: 5000

Shape antes: (5000, 17)  ->  S

,id_venta,pedido_id,fecha_venta,id_cliente,ciudad,canal,id_tienda,categoria,producto,cantidad,precio_unitario,descuento_pct,valor_bruto,valor_descuento,valor_neto,medio_pago,calificacion_cliente
0,1,PED000001,2026-01-18 22:06:15,CLI00011,Bogotá D.C.,Aplicación móvil,NaN,Deportes,Banda caminadora,1,2199900.0,10.0,2199900.0,219990.0,1979910.0,Tarjeta crédito,NaN
1,2,PED000002,2026-04-25 21:07:01,CLI00802,Cali,Página web,NaN,Deportes,Mancuernas ajustables,1,399900.0,10.0,399900.0,39990.0,359910.0,Tarjeta crédito,NaN
2,3,PED000003,2026-03-28 02:39:43,CLI00255,Medellín,Página web,NaN,Tecnología,Tablet 10 pulgadas,1,899900.0,10.0,899900.0,89990.0,809910.0,Tarjeta crédito,4.0
3,4,PED000004,2026-04-30 12:35:40,CLI00383,Medellín,Aplicación móvil,NaN,Electrodomésticos,Freidora de aire,2,472400.0,15.0,944800.0,141720.0,803080.0,PSE,5.0
4,5,PED000005,2026-04-04 00:18:12,CLI01286,Bogotá D.C.,Aplicación móvil,NaN,Deportes,Balón fútbol,1,119900.0,0.0,119900.0,0.0,119900.0,Tarjeta crédito,NaN


## Capa Silver — `df_logistica_transformado`

Decisiones de limpieza aplicadas (ver `silver.limpiar_logistica`):

1. Se eliminan filas completamente duplicadas y duplicados de `evento_id` (conservando el primero).
2. Eventos sin `pedido_id` se descartan (no se pueden vincular a ninguna venta).
3. `fecha_evento` se parsea a `datetime`; inválidas quedan como `NaT`.
4. `estado_evento`, `ciudad_destino`, `transportadora` se estandarizan contra catálogos controlados.
5. `incidencia` nulo → `"SIN INCIDENCIA"` (el nulo aquí tiene un significado de negocio real: no ocurrió ningún incidente).
6. `costo_envio`: se limpia el formato de texto tipo `"$ 45.000"` y se convierte a numérico; los nulos resultantes se imputan con la **mediana por transportadora** (variable monetaria con outliers).
7. `tiempo_etapa_horas` negativo (error de captura) → se lleva a `NaN` y se imputa con la **mediana por estado_evento** (cada etapa tiene una duración típica distinta).

In [5]:
df_logistica_transformado, reporte_logistica = silver.limpiar_logistica(df_logistica)

print("Reporte de limpieza df_logistica -> df_logistica_transformado:")
for k, v in reporte_logistica.items():
    print(f"  {k}: {v}")

print(f"\nShape antes: {df_logistica.shape}  ->  Shape despues: {df_logistica_transformado.shape}")
df_logistica_transformado.head()

Reporte de limpieza df_logistica -> df_logistica_transformado:
  filas_entrada: 50000
  duplicados_completos_eliminados: 99
  duplicados_evento_id_eliminados: 1
  filas_sin_pedido_id_eliminadas: 0
  fechas_evento_invalidas: 0
  estandarizacion_estado_evento: {'valores_originales_no_nulos': 49900, 'valores_reconocidos': 19965, 'valores_no_reconocidos': 29935, 'detalle_no_reconocidos': {'Listo para despacho': 4995, 'Pago aprobado': 4990, 'En alistamiento': 4990, 'Empacado': 4990, 'Orden enviada a logística': 4988, 'En ruta de entrega': 4982}}
  estandarizacion_ciudad_destino: {'valores_originales_no_nulos': 49855, 'valores_reconocidos': 49855, 'valores_no_reconocidos': 0, 'detalle_no_reconocidos': {}}
  estandarizacion_transportadora: {'valores_originales_no_nulos': 19876, 'valores_reconocidos': 15879, 'valores_no_reconocidos': 3997, 'detalle_no_reconocidos': {'Inter Rapidísimo': 3997}}
  incidencia_nulos_a_sin_incidencia: 49389
  estandarizacion_incidencia: {'valores_originales_no_nulos

,evento_id,pedido_id,fecha_evento,estado_evento,centro_logistico,ciudad_destino,transportadora,numero_guia,fecha_prometida_entrega,tiempo_etapa_horas,costo_envio,incidencia,observacion,Unnamed: 13
0,1,PED000001,2026-01-18 22:06:15,Pedido Recibido,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,3.06,8900,SIN INCIDENCIA,NaN,"=AI("""")"
1,2,PED000001,2026-01-18 22:36:53,NO INFORMADO,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,0.51,8900,SIN INCIDENCIA,NaN,NaN
2,3,PED000001,2026-01-18 23:31:22,NO INFORMADO,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,0.91,8900,SIN INCIDENCIA,NaN,NaN
3,4,PED000001,2026-01-19 02:03:55,NO INFORMADO,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,2.54,8900,SIN INCIDENCIA,NaN,NaN
4,5,PED000001,2026-01-19 06:14:51,NO INFORMADO,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,4.18,8900,SIN INCIDENCIA,NaN,NaN


### Validación: perfil de nulos antes vs. después de Silver

Se compara el perfil de nulos original contra el transformado para confirmar que las columnas tratadas (según lo documentado arriba) efectivamente mejoraron, y que las que se dejaron nulas a propósito (`calificacion_cliente`, fechas inválidas) siguen siendo nulas.

In [7]:
print("Nulos ANTES (df_ventas):")
display(quality.perfil_nulos(df_ventas))
print("\nNulos DESPUES (df_ventas_transformado):")
display(quality.perfil_nulos(df_ventas_transformado))

print("\nNulos ANTES (df_logistica):")
display(quality.perfil_nulos(df_logistica))
print("\nNulos DESPUES (df_logistica_transformado):")
display(quality.perfil_nulos(df_logistica_transformado))

# Persistencia de la capa Silver
df_ventas_transformado.to_parquet(PROJECT_ROOT / "data/silver/ventas_transformado.parquet", index=False)
df_logistica_transformado.to_parquet(PROJECT_ROOT / "data/silver/logistica_transformado.parquet", index=False)

Nulos ANTES (df_ventas):


,columna,cantidad_nulos,pct_nulos
0,id_tienda,3501,70.02
1,calificacion_cliente,2172,43.44
2,id_venta,0,0.00
3,id_cliente,0,0.00
4,pedido_id,0,0.00
5,ciudad,0,0.00
6,canal,0,0.00
7,categoria,0,0.00
8,fecha_venta,0,0.00
9,producto,0,0.00



Nulos DESPUES (df_ventas_transformado):


,columna,cantidad_nulos,pct_nulos
0,id_tienda,3501,70.02
1,calificacion_cliente,2172,43.44
2,id_venta,0,0.00
3,id_cliente,0,0.00
4,pedido_id,0,0.00
5,ciudad,0,0.00
6,canal,0,0.00
7,categoria,0,0.00
8,fecha_venta,0,0.00
9,producto,0,0.00



Nulos ANTES (df_logistica):


,columna,cantidad_nulos,pct_nulos
0,Unnamed: 13,49999,100.00
1,incidencia,49489,98.98
2,observacion,48277,96.55
3,transportadora,30088,60.18
4,numero_guia,30068,60.14
5,tiempo_etapa_horas,5000,10.00
6,centro_logistico,90,0.18
7,fecha_evento,65,0.13
8,fecha_prometida_entrega,55,0.11
9,ciudad_destino,45,0.09



Nulos DESPUES (df_logistica_transformado):


,columna,cantidad_nulos,pct_nulos
0,Unnamed: 13,49899,100.00
1,observacion,48183,96.56
2,transportadora,30024,60.17
3,numero_guia,30004,60.13
4,centro_logistico,90,0.18
5,fecha_evento,65,0.13
6,fecha_prometida_entrega,55,0.11
7,ciudad_destino,45,0.09
8,evento_id,0,0.00
9,pedido_id,0,0.00


## Capa Gold — integración por `pedido_id` + datamarts

`df_logistica_transformado` tiene múltiples eventos por `pedido_id`; se colapsan a una fila resumen por pedido (estado actual, transportadora, ciclo logístico total, costo de envío, si tuvo incidencia) y se integra con `df_ventas_transformado` mediante un `merge` por `pedido_id`.

In [8]:
df_gold_pedidos, reporte_gold = gold.construir_gold_pedidos(
    df_ventas_transformado, df_logistica_transformado
)
print("Reporte de integracion Gold:")
for k, v in reporte_gold.items():
    print(f"  {k}: {v}")

gold.guardar_gold(df_gold_pedidos, "pedidos_360")

marts = gold.construir_marts(df_gold_pedidos)
for nombre_mart, df_mart in marts.items():
    gold.guardar_gold(df_mart, nombre_mart)
    print(f"\n--- Mart: {nombre_mart} ---")
    display(df_mart.head(10))

df_gold_pedidos.head()

2026-09-30 22:28:17 | INFO     | kaismart_etl.transform.gold | Gold guardado: D:\Cursos\Phyton\kaismart-etl\data\gold\pedidos_360.csv (5000 registros)
2026-09-30 22:28:17 | INFO     | kaismart_etl.transform.gold | Gold guardado: D:\Cursos\Phyton\kaismart-etl\data\gold\ventas_por_ciudad_categoria.csv (30 registros)


Reporte de integracion Gold:
  pedidos_ventas: 5000
  pedidos_con_logistica: 5000
  pedidos_sin_eventos_logisticos: 0
  eventos_logisticos_sin_venta_asociada: 0

--- Mart: ventas_por_ciudad_categoria ---


,ciudad,categoria,valor_neto
5,Bogotá D.C.,Deportes,273790785.0
9,Bogotá D.C.,Tecnología,240385895.0
8,Bogotá D.C.,Oficina,216705250.0
15,Cali,Deportes,183577540.0
20,Medellín,Deportes,162374510.0
7,Bogotá D.C.,Hogar,157488760.0
19,Cali,Tecnología,155611235.0
6,Bogotá D.C.,Electrodomésticos,145291150.0
24,Medellín,Tecnología,139310545.0
23,Medellín,Oficina,118062210.0


2026-09-30 22:28:17 | INFO     | kaismart_etl.transform.gold | Gold guardado: D:\Cursos\Phyton\kaismart-etl\data\gold\pedidos_por_estado_logistico.csv (5 registros)



--- Mart: pedidos_por_estado_logistico ---


,estado_actual,pedidos
0,Entregado,4931
1,NO INFORMADO,45
2,Pedido Recibido,10
3,Despachado,7
4,En Tránsito,7


2026-09-30 22:28:17 | INFO     | kaismart_etl.transform.gold | Gold guardado: D:\Cursos\Phyton\kaismart-etl\data\gold\kpis_por_transportadora.csv (5 registros)



--- Mart: kpis_por_transportadora ---


,transportadora,pedidos,costo_envio_promedio,dias_ciclo_promedio,pct_con_incidencia
0,Coordinadora,1027,13701.31,1.50,0.1
1,Envía,1059,13509.45,1.50,0.1
2,NO INFORMADO,1007,13721.29,1.52,0.1
3,Servientrega,950,13531.32,1.49,0.1
4,TCC,957,13588.9,1.50,0.1


,id_venta,pedido_id,fecha_venta,id_cliente,ciudad,canal,id_tienda,categoria,producto,cantidad,precio_unitario,descuento_pct,valor_bruto,valor_descuento,valor_neto,medio_pago,calificacion_cliente,cantidad_eventos,estado_actual,fecha_primer_evento,fecha_ultimo_evento,transportadora,ciudad_destino,tiempo_total_horas,costo_envio,tuvo_incidencia,dias_ciclo_logistico
0,1,PED000001,2026-01-18 22:06:15,CLI00011,Bogotá D.C.,Aplicación móvil,NaN,Deportes,Banda caminadora,1,2199900.0,10.0,2199900.0,219990.0,1979910.0,Tarjeta crédito,NaN,10,Entregado,2026-01-18 22:06:15,2026-01-20 14:18:58,TCC,Bogotá D.C.,43.26,8900,False,1.675498
1,2,PED000002,2026-04-25 21:07:01,CLI00802,Cali,Página web,NaN,Deportes,Mancuernas ajustables,1,399900.0,10.0,399900.0,39990.0,359910.0,Tarjeta crédito,NaN,10,Entregado,2026-04-25 21:07:01,2026-04-27 11:00:48,Envía,Cali,40.95,8900,False,1.579016
2,3,PED000003,2026-03-28 02:39:43,CLI00255,Medellín,Página web,NaN,Tecnología,Tablet 10 pulgadas,1,899900.0,10.0,899900.0,89990.0,809910.0,Tarjeta crédito,4.0,10,Entregado,2026-03-28 02:39:43,2026-03-29 22:55:26,NO INFORMADO,Medellín,47.32,13900,False,1.844248
3,4,PED000004,2026-04-30 12:35:40,CLI00383,Medellín,Aplicación móvil,NaN,Electrodomésticos,Freidora de aire,2,472400.0,15.0,944800.0,141720.0,803080.0,PSE,5.0,10,Entregado,2026-04-30 12:35:40,2026-05-03 08:20:40,Envía,Medellín,70.81,13900,False,2.822917
4,5,PED000005,2026-04-04 00:18:12,CLI01286,Bogotá D.C.,Aplicación móvil,NaN,Deportes,Balón fútbol,1,119900.0,0.0,119900.0,0.0,119900.0,Tarjeta crédito,NaN,10,Entregado,2026-04-04 00:18:12,2026-04-05 08:43:56,Coordinadora,Bogotá D.C.,35.49,8900,False,1.351204


## Parte 8. Automatización

Todo este flujo (extracción → Bronze → Silver → Gold) está encapsulado en `kaismart_etl.pipeline.run_etl()`. La automatización periódica con la librería `schedule` está implementada en `kaismart_etl/scheduler.py` y se ejecuta con:

```bash
python scripts/run_scheduler.py
```

Esto corre el pipeline inmediatamente y luego cada `SCHEDULE_INTERVAL_MINUTES` minutos (configurable en `.env`), de forma indefinida, hasta detenerlo con `Ctrl+C`.